# Construction video analysis
Run setup/model downloads once on an L4, then rerun either upload/inference/plot workflow. Both models stay in memory. Short-video timing bars measure the whole submitted set and exclude model download/loading. Option scores are not calibrated accuracy probabilities.

## 1. Installs

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==5.21.0', 'accelerate', 'safetensors', 'einops',
    'opencv-python-headless==4.11.0.86'], check=True)


## 2. Model downloads
Imports, shared prompt, and one-time model loading.

In [ ]:
import json, time, math, tempfile
from pathlib import Path
import cv2
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from IPython.display import display, JSON
from google.colab import files
from transformers import AutoModel, AutoProcessor, AutoModelForImageTextToText

assert torch.cuda.is_available(), 'Select a GPU under Runtime > Change runtime type.'
print('GPU:', torch.cuda.get_device_name(0))
STATE = 'Silent chronological POV footage from construction equipment. Classify only observable action. Equipment type alone is not evidence of activity.'
QUESTION = 'Which option best describes the dominant observable activity in this video?'
LABELS = {
    'standing_still': 'Vehicle and attachment stay still, ignoring minor vibration and foliage.',
    'driving': 'The vehicle travels across the site; moving the attachment alone is not driving.',
    'digging': 'The tool excavates, scrapes or scoops soil or material.',
    'other': 'Unclear, mixed or another action, including tool movement without clear digging.'
}
LETTERS = list('ABCD')
COLORS = {'standing_still':'#258c76', 'driving':'#3478b7', 'digging':'#dd8c26', 'other':'#9299a5'}


In [ ]:
started = time.perf_counter()
d3 = AutoModel.from_pretrained('vllm-sr/d3-nano',
    revision='8198c9f763b046605b2f4827b4999971fa2c92c4',
    trust_remote_code=True, device='cuda:0', batch_size=1)
torch.cuda.synchronize()
print(f'd3 downloaded/loaded in {time.perf_counter()-started:.1f}s')


In [ ]:
started = time.perf_counter()
lfm_id, revision = 'LiquidAI/LFM3-VL-2B', '71d8e745d97c224984e0f20412f532a5ce9dfd3e'
processor = AutoProcessor.from_pretrained(lfm_id, revision=revision, trust_remote_code=True)
lfm = AutoModelForImageTextToText.from_pretrained(lfm_id, revision=revision,
    trust_remote_code=True, dtype=torch.bfloat16, device_map={'':'cuda:0'}, attn_implementation='sdpa').eval()
processor.tokenizer.padding_side = 'left'
letter_tokens = [processor.tokenizer.encode(c, add_special_tokens=False) for c in LETTERS]
assert all(len(ids)==1 for ids in letter_tokens), 'Expected single-token answer letters.'
TOKEN_IDS = [ids[0] for ids in letter_tokens]
torch.cuda.synchronize()
print(f'LFM downloaded/loaded in {time.perf_counter()-started:.1f}s')


In [ ]:
def sample_video(path):
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if not cap.isOpened() or fps <= 0 or total < 4:
        cap.release()
        raise ValueError(f'Invalid or too-short video: {path}')
    indices = np.linspace(0, total - 1, min(max(int(total / fps * 2), 4), 32, total)).round().astype(int).tolist()
    wanted, frames, index = set(indices), [], 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if index in wanted:
            h, w = frame.shape[:2]
            scale = min(1.0, 512 / max(h, w))
            frame = cv2.resize(frame, (round(w * scale), round(h * scale)))
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        index += 1
    cap.release()
    if len(frames) != len(indices):
        raise ValueError(f'Incomplete video: {path}')
    if len(frames) % 2:
        frames.append(frames[-1])
        indices.append(indices[-1])
    metadata = {'fps': fps, 'total_num_frames': total, 'duration': total/fps,
                'frames_indices': indices, 'width': frames[0].shape[1], 'height': frames[0].shape[0]}
    return np.stack(frames), metadata


def decision_row(item, scores):
    scores = {key: float(scores[key]) for key in LABELS}
    if not all(np.isfinite(value) and value >= 0 for value in scores.values()):
        raise ValueError('Invalid option scores')
    return {'video': item['name'], 'label': max(scores, key=scores.get), 'option_scores': scores}


def predict_d3_batch(items):
    # This is an application batch: the native API decides each clip independently.
    # Its videos=[...] parameter does NOT mean independent-request batching.
    torch.cuda.synchronize()
    start = time.perf_counter()
    rows = []
    for i, item in enumerate(items):
        response = d3.system_one(state=STATE, videos=[str(item['path'])], questions={
            'activity': {'type': 'choice', 'instructions': QUESTION, 'criteria': LABELS}})
        answer = response['answers']['activity']
        if 'probabilities' not in answer:
            raise RuntimeError(f'd3 failed for {item["name"]}: {answer}')
        rows.append(decision_row(item, answer['probabilities']))
        print(f'\rd3: {i+1}/{len(items)}', end='', flush=True)
    torch.cuda.synchronize()
    print()
    return {'model': 'd3', 'mode': 'independent native calls',
            'processing_seconds': time.perf_counter()-start, 'videos': rows}


def predict_lfm_batch(items):
    torch.cuda.synchronize()
    start = time.perf_counter()
    prepared = [sample_video(item['path']) for item in items]
    # Exactly the same state, question, option order, and definitions as d3.
    options = '\n'.join(f'{letter}. {key}: {LABELS[key]}' for letter, key in zip(LETTERS, LABELS))
    prompt = STATE + '\n' + QUESTION + '\n' + options + '\nAnswer with exactly one letter A, B, C, or D.'
    texts = [processor.apply_chat_template(
        [{'role': 'user', 'content': [{'type': 'video'}, {'type': 'text', 'text': prompt}]}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False) for _ in items]
    # Explicitly associate ONE video with EACH prompt, avoiding combined-video context.
    inputs = processor(text=texts, videos=[[frames] for frames, _ in prepared],
                       video_metadata=[metadata for _, metadata in prepared],
                       do_sample_frames=False, padding=True, return_tensors='pt')
    inputs = {k: v.to('cuda:0') if hasattr(v, 'to') else v for k, v in inputs.items() if k != 'video_metadata'}
    with torch.inference_mode():
        logits = lfm(**inputs, use_cache=False, logits_to_keep=1).logits[:, -1]
        probabilities = torch.softmax(logits[:, TOKEN_IDS].float(), dim=-1).cpu().numpy()
    torch.cuda.synchronize()
    elapsed = time.perf_counter()-start
    if probabilities.shape != (len(items), len(LABELS)):
        raise RuntimeError('LFM batch output does not match the independent inputs')
    rows = [decision_row(item, dict(zip(LABELS, row))) for item, row in zip(items, probabilities)]
    return {'model': 'lfm', 'mode': 'tensor batch', 'processing_seconds': elapsed, 'videos': rows}


## 3. Short videos — uploads
Upload 1–3 clips. Rerun this section onward for another set; keep the loaded models.

In [ ]:
short_reports = None
uploaded = files.upload()
assert 1 <= len(uploaded) <= 3, 'Upload 1–3 short videos.'
short_dir = Path(tempfile.mkdtemp(prefix='short_actions_'))
short_items = []
for i, (name, content) in enumerate(uploaded.items()):
    path = short_dir/f'clip_{i}.mp4'
    path.write_bytes(content)
    short_items.append({'name': name, 'path': path})
del uploaded
print([item['name'] for item in short_items])


## 4. Short videos — model inference
Submit the complete list to d3 first, then LFM. The current d3 API requires independent native calls inside its batch adapter; putting all videos in one request would produce a combined-context answer. LFM uses one tensor batch, with one video explicitly assigned to each prompt. No download or model loading occurs here.

In [ ]:
short_reports = None
first_model = predict_d3_batch(short_items)
second_model = predict_lfm_batch(short_items)
short_reports = [first_model, second_model]
print('Finished both models.')


## 5. Short videos — plots
Decisions are categorical: use a table and option-score heatmaps. The timing chart is a simple bar chart: total processing time for the same N clips, model versus model. Both timers include video preparation and inference; the first call can include GPU warm-up.

In [ ]:
assert short_reports is not None, 'Run short-video inference first.'
rows = [{'video': row['video'], 'model': report['model'], 'decision': row['label']}
        for report in short_reports for row in report['videos']]
display(pd.DataFrame(rows).pivot(index='video', columns='model', values='decision'))
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for ax, report in zip(axes[:2], short_reports):
    scores = np.array([[row['option_scores'][label] for label in LABELS] for row in report['videos']])
    ax.imshow(scores, vmin=0, vmax=1, cmap='Blues', aspect='auto')
    ax.set_xticks(range(len(LABELS)), [label.replace('_',' ') for label in LABELS], rotation=25, ha='right')
    ax.set_yticks(range(len(report['videos'])), [row['video'] for row in report['videos']])
    ax.set_title(f"{report['model']}: option scores")
    for i in range(scores.shape[0]):
        for j in range(scores.shape[1]):
            ax.text(j, i, f'{scores[i,j]:.2f}', ha='center', va='center', color='white' if scores[i,j]>.55 else 'black')
times = [report['processing_seconds'] for report in short_reports]
axes[2].bar(['d3','lfm'], times, color=['#3478b7','#dd8c26'])
axes[2].bar_label(axes[2].containers[0], fmt='%.2f s')
axes[2].set_ylabel('Seconds'); axes[2].set_ylim(0, max(times)*1.2)
axes[2].set_title(f'Processing time: all {len(short_items)} clips')
plt.tight_layout(); plt.show()
short_output = {'state':STATE, 'question':QUESTION, 'labels':LABELS, 'models':short_reports}
display(JSON(short_output, expanded=False))
short_json = short_dir/'short_video_results.json'
short_json.write_text(json.dumps(short_output, indent=2))
files.download(str(short_json))


## 6. Long video — upload
Upload one recording. The existing models and inference helpers are reused.

In [ ]:
long_output = None
uploaded = files.upload()
assert len(uploaded)==1, 'Upload one long video.'
long_name, content = next(iter(uploaded.items()))
long_dir = Path(tempfile.mkdtemp(prefix='long_actions_'))
long_input = long_dir/'input_video.mp4'
long_input.write_bytes(content)
del uploaded, content
probe = subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','json',str(long_input)],
                       check=True, capture_output=True, text=True)
long_duration = float(json.loads(probe.stdout)['format']['duration'])
assert math.isfinite(long_duration) and long_duration >= 1, 'Use a video at least one second long.'
print(f'{long_name}: {long_duration:.2f}s')


## 7. Long video — models inference
Split into 10-second windows, then reuse the same prompt and adapters. LFM uses batches of three; d3 produces independent decisions. A final fragment shorter than one second joins the preceding window.

In [ ]:
long_output = None
WINDOW_SECONDS, LFM_BATCH_SIZE = 10, 3
starts = list(range(0, math.ceil(long_duration), WINDOW_SECONDS))
if len(starts)>1 and long_duration-starts[-1]<1:
    starts.pop()
segments, long_items = [], []
for i, start in enumerate(starts):
    end = starts[i+1] if i+1<len(starts) else long_duration
    path = long_dir/f'clip_{i:05d}.mp4'
    subprocess.run(['ffmpeg','-v','error','-y','-ss',str(start),'-i',str(long_input),
                    '-t',str(end-start),'-map','0:v:0','-an','-c:v','libx264',
                    '-preset','veryfast','-crf','20','-pix_fmt','yuv420p',str(path)], check=True)
    segments.append({'start_seconds':start,'end_seconds':end})
    long_items.append({'name':path.name,'path':path})
print(f'Prepared {len(long_items)} intervals.')
long_d3 = predict_d3_batch(long_items)
long_lfm_rows, lfm_batch_times = [], []
for i in range(0, len(long_items), LFM_BATCH_SIZE):
    batch = predict_lfm_batch(long_items[i:i+LFM_BATCH_SIZE])
    long_lfm_rows.extend(batch['videos'])
    lfm_batch_times.append(batch['processing_seconds'])
    print(f'LFM: {min(i+LFM_BATCH_SIZE,len(long_items))}/{len(long_items)}')
assert len(segments)==len(long_d3['videos'])==len(long_lfm_rows)
for segment, a, b in zip(segments, long_d3['videos'], long_lfm_rows):
    segment.update(d3=a, lfm=b)
long_results = segments
long_output = {'video':long_name,'duration_seconds':long_duration,'window_seconds':WINDOW_SECONDS,
               'state':STATE,'question':QUESTION,'labels':LABELS,'segments':long_results,
               'processing_seconds':{'d3':long_d3['processing_seconds'],'lfm':sum(lfm_batch_times)}}


## 8. Long video — models plot
A categorical timeline shows the action assigned to each interval by each model.

In [ ]:
assert long_output is not None, 'Run long-video inference first.'
fig, ax = plt.subplots(figsize=(16, 3.5))
for y, model in [(1,'d3'),(0,'lfm')]:
    for row in long_results:
        ax.broken_barh([(row['start_seconds'],row['end_seconds']-row['start_seconds'])],
                       (y-.32,.64), facecolors=COLORS[row[model]['label']], edgecolors='white', linewidth=.4)
ax.set_yticks([1,0],['d3','lfm']); ax.set_xlim(0,long_duration)
ax.set_xlabel('Video time (seconds)'); ax.set_title(f'Action labels: {long_name}')
ax.legend(handles=[Patch(facecolor=color,label=label.replace('_',' ')) for label,color in COLORS.items()],
          loc='upper center',bbox_to_anchor=(.5,-.2),ncol=4)
plt.tight_layout(); plt.show()
display(JSON(long_output, expanded=False))
long_json_path = long_dir/'long_video_results.json'
long_json_path.write_text(json.dumps(long_output,indent=2))
files.download(str(long_json_path))


## 9. Long video — motion graph

Reuse the existing repository implementation and accepted default settings on the original long recording. The bundled sources below avoid an extra GitHub download or login.


In [ ]:
# Exact repository sources bundled for Colab; no GitHub login is required.
import importlib.util
import sys

MOTION_SOURCE_COMMIT = "c414546ade8306322d66cbb9c2981164b748fc06"
motion_code_dir = long_dir / 'motion_code'
motion_code_dir.mkdir(exist_ok=True)
motion_sources = {
    'smoothing.py': "\"\"\"Presentation smoothing for one-second motion scores; raw scores stay unchanged.\n\nSupply neighboring scores across chunk boundaries, smooth, then trim context.\nNone/NaN marks a gap and is preserved. At true run boundaries, repeat the edge\nscore instead of inserting zeros. Weights operate over seconds, not video frames.\n\"\"\"\nimport numpy as np\n\nGENTLE = (1, 2, 1)\nSTRONGER = (1, 2, 3, 2, 1)\n\n\ndef smooth_scores(scores, weights=GENTLE):\n    values = np.asarray(scores, dtype=float)\n    kernel = np.asarray(weights, dtype=float)\n    if values.ndim != 1 or kernel.ndim != 1:\n        raise ValueError('Scores and weights must be one-dimensional')\n    if not len(kernel) or len(kernel) % 2 != 1:\n        raise ValueError('Weights must have a positive odd length')\n    if not np.all(np.isfinite(kernel)) or np.any(kernel < 0) or kernel.sum() <= 0:\n        raise ValueError('Weights must be finite, nonnegative, with positive sum')\n    if np.any(np.isinf(values)):\n        raise ValueError('Use None or NaN for gaps, not infinity')\n    valid = np.isfinite(values)\n    if np.any((values[valid] < 0) | (values[valid] > 1)):\n        raise ValueError('Motion scores must be between zero and one')\n    result = np.full(values.shape, np.nan)\n    kernel = kernel / kernel.sum()\n    radius = len(kernel) // 2\n    # Process each continuous run independently, preserving missing coverage.\n    edges = np.diff(np.r_[False, valid, False].astype(int))\n    for start, end in zip(np.flatnonzero(edges == 1), np.flatnonzero(edges == -1)):\n        padded = np.pad(values[start:end], (radius, radius), mode='edge')\n        result[start:end] = np.convolve(padded, kernel, mode='valid')\n    return result\n",
    'motion_graph.py': "\"\"\"Construction POV motion graph: CPU detection, fixed calibration, temporal smoothing.\n\nRun: python motion_graph.py video.mp4 --output result.json --plot graph.png\nOnly NumPy/OpenCV are required for analysis. Matplotlib is optional for exports.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nfrom collections import deque\nfrom dataclasses import asdict, dataclass\nimport hashlib\nimport io\nimport json\nimport math\nfrom pathlib import Path\nimport sys\nimport time\n\nimport cv2\nimport numpy as np\nfrom smoothing import GENTLE, STRONGER, smooth_scores\n\nALGORITHM_VERSION = 'contech-motion-1.0.0'\nSCHEMA_VERSION = '1.0'\nKERNELS = {'none': (1,), 'gentle': GENTLE, 'stronger': STRONGER}\n\n\n@dataclass(frozen=True)\nclass Config:\n    width: int = 320\n    sample_fps: int = 5\n    lag_samples: int = 3\n    blur_size: int = 5\n    pixel_threshold: float = 12.0\n    min_component_fraction: float = 0.001\n    area_deadband: float = 0.01\n    calibration_beta: float = 0.15\n    smoothing: str = 'stronger'\n\n    def __post_init__(self):\n        if any(not isinstance(v, int) or isinstance(v, bool) or v < 1\n               for v in (self.width, self.sample_fps, self.lag_samples, self.blur_size)):\n            raise ValueError('Dimensions and sampling parameters must be positive integers')\n        if self.width < 32 or self.blur_size % 2 != 1:\n            raise ValueError('Width must be >=32; blur_size must be odd')\n        for name in ('pixel_threshold', 'min_component_fraction', 'area_deadband', 'calibration_beta'):\n            if not math.isfinite(getattr(self, name)):\n                raise ValueError(f'{name} must be finite')\n        if not 0 < self.pixel_threshold < 255:\n            raise ValueError('pixel_threshold must be between 0 and 255')\n        if not 0 <= self.min_component_fraction < 1 or not 0 <= self.area_deadband < 1:\n            raise ValueError('Area fractions must be in [0,1)')\n        if self.calibration_beta <= 0 or self.smoothing not in KERNELS:\n            raise ValueError('Invalid calibration beta or smoothing mode')\n\n\ndef prepare_frame(frame, config):\n    height, width = frame.shape[:2]\n    target = min(width, config.width)\n    resized = cv2.resize(frame, (target, max(1, round(height * target / width))),\n                         interpolation=cv2.INTER_AREA)\n    gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)\n    return cv2.GaussianBlur(gray, (config.blur_size, config.blur_size), 0)\n\n\ndef measure_change(previous, current, config):\n    if previous.shape != current.shape:\n        raise ValueError('Video dimensions changed during decoding')\n    difference = current.astype(np.float32) - previous.astype(np.float32)\n    offset = float(np.median(difference))\n    difference -= offset\n    mask = (np.abs(difference) > config.pixel_threshold).astype(np.uint8)\n    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))\n    _, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)\n    keep = stats[:, cv2.CC_STAT_AREA] >= max(1, math.ceil(mask.size * config.min_component_fraction))\n    keep[0] = False\n    area = float(keep[labels].mean())\n    detector = float(np.clip((area - config.area_deadband) / (1 - config.area_deadband), 0, 1))\n    calibrated = (1 + config.calibration_beta) * detector / (detector + config.calibration_beta)\n    return {'changed_area_fraction': area, 'detector_score': detector,\n            'calibrated_score': float(calibrated), 'brightness_offset': offset}\n\n\ndef _sample_frames(path, config, metadata, stop_seconds=None):\n    cap = cv2.VideoCapture(str(path))\n    if not cap.isOpened():\n        raise ValueError(f'Cannot open video: {path}')\n    fps = float(cap.get(cv2.CAP_PROP_FPS))\n    if not math.isfinite(fps) or fps < config.sample_fps:\n        cap.release()\n        raise ValueError('Source must report a valid frame rate >= sample_fps')\n    reported_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))\n    metadata.update(fps_reported=fps, width=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),\n                    height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)), decoded_frames=0,\n                    reached_eof=False)\n    origin = None\n    previous_pts = -math.inf\n    tick = 0\n    try:\n        while True:\n            ok, frame = cap.read()\n            if not ok:\n                metadata['reached_eof'] = True\n                if reported_frames > 0 and metadata['decoded_frames'] != reported_frames:\n                    raise ValueError('Decoded frame count differs from container metadata; validate input')\n                break\n            pts = float(cap.get(cv2.CAP_PROP_POS_MSEC)) / 1000\n            if not math.isfinite(pts) or pts <= previous_pts:\n                raise ValueError('Missing or non-monotonic decoded timestamps')\n            interval = pts - previous_pts if math.isfinite(previous_pts) else 1 / fps\n            previous_pts = pts\n            if origin is None:\n                origin = pts\n                metadata['first_decoded_timestamp_seconds'] = pts\n            relative = pts - origin\n            metadata['decoded_frames'] += 1\n            metadata['decoded_through_seconds'] = relative + interval\n            if stop_seconds is not None and relative >= stop_seconds - 1e-6:\n                break\n            target = tick / config.sample_fps\n            if relative + 1e-6 >= target:\n                if relative - target > 1 / config.sample_fps + 1e-6:\n                    raise ValueError('Timestamp gap exceeds sampling interval; split at recording gaps')\n                yield target, relative, prepare_frame(frame, config)\n                tick += 1\n    finally:\n        cap.release()\n\n\ndef _window_bound(value, name):\n    if value is not None and (not math.isfinite(value) or value < 0 or not float(value).is_integer()):\n        raise ValueError(f'{name} must be a finite nonnegative whole number of seconds')\n\n\ndef _nullable(value):\n    return float(value) if np.isfinite(value) else None\n\n\ndef analyze_video(path, config=None, *, start_seconds=0, end_seconds=None, include_samples=False):\n    \"\"\"Analyze a validated local video, retaining neighboring footage for smoothing.\n\n    Bounds refer to the input file, relative to its first decoded frame. Requested\n    bounds must be whole seconds. Output timestamps are relative to start_seconds.\n    Default: selected smooth calibration plus five-second temporal smoothing.\n    \"\"\"\n    config = config or Config()\n    _window_bound(start_seconds, 'start_seconds')\n    _window_bound(end_seconds, 'end_seconds')\n    if end_seconds is not None and end_seconds <= start_seconds:\n        raise ValueError('end_seconds must be greater than start_seconds')\n    path = Path(path)\n    if not path.is_file():\n        raise ValueError(f'Video is not a local file: {path}')\n    started = time.perf_counter()\n    cv2.setNumThreads(1)\n    radius = len(KERNELS[config.smoothing]) // 2\n    stop = None if end_seconds is None else end_seconds + radius\n    metadata = {}\n    history = deque(maxlen=config.lag_samples)\n    samples = []\n    for tick, pts, frame in _sample_frames(path, config, metadata, stop):\n        if len(history) == config.lag_samples:\n            prev_pts, previous = history[0]\n            row = measure_change(previous, frame, config)\n            row.update(input_time_seconds=round(tick, 6),\n                       decoded_time_seconds=round(pts, 6),\n                       comparison_interval_seconds=round(pts - prev_pts, 6))\n            samples.append(row)\n        history.append((pts, frame))\n    if not samples:\n        raise ValueError('Insufficient valid frames: at least 0.6s of comparison context is needed')\n    available_end = round(metadata['decoded_through_seconds'], 9)\n    if end_seconds is not None and metadata['reached_eof'] and end_seconds > available_end + 1e-6:\n        raise ValueError('Requested end_seconds is beyond decoded video duration')\n    output_end = float(end_seconds) if end_seconds is not None else available_end\n    if start_seconds >= output_end:\n        raise ValueError('start_seconds is outside the decoded video')\n\n    # Bucket by input clock, including context. Apply calibration before averaging.\n    buckets = {}\n    for row in samples:\n        buckets.setdefault(math.floor(row['input_time_seconds']), []).append(row)\n    count = math.ceil(min(available_end, stop) if stop is not None else available_end)\n    scores = np.full(count, np.nan)\n    for index, members in buckets.items():\n        if index < count:\n            scores[index] = np.mean([m['calibrated_score'] for m in members])\n    smoothed = smooth_scores(scores, KERNELS[config.smoothing])\n    rows = []\n    first = int(start_seconds)\n    for index in range(first, math.ceil(output_end)):\n        members = buckets.get(index, [])\n        end = min(index + 1, output_end)\n        expected = sum(index <= k / config.sample_fps < end - 1e-9\n                       for k in range(index * config.sample_fps, (index + 1) * config.sample_fps))\n        padded = index - radius < 0 or index + radius >= len(scores)\n        if not padded:\n            padded = not np.isfinite(scores[index-radius:index+radius+1]).all()\n        row = {'start_seconds': round(index - start_seconds, 9),\n               'end_seconds': round(end - start_seconds, 9),\n               'input_start_seconds': index,\n               'score': _nullable(smoothed[index]),\n               'calibrated_score': _nullable(scores[index]),\n               'detector_score': float(np.mean([m['detector_score'] for m in members])) if members else None,\n               'changed_area_fraction': float(np.mean([m['changed_area_fraction'] for m in members])) if members else None,\n               'sample_count': len(members), 'expected_sample_count': expected,\n               'coverage_fraction': len(members) / expected if expected else 0.0,\n               'smoothing_edge_padded': bool(padded)}\n        rows.append(row)\n    target_samples = [s for s in samples if start_seconds <= s['input_time_seconds'] < output_end]\n    if not target_samples:\n        raise ValueError('No valid comparison measurements in the requested window')\n    valid = [r['score'] for r in rows if r['score'] is not None]\n    digest = hashlib.sha256()\n    with path.open('rb') as source:\n        for block in iter(lambda: source.read(1024 * 1024), b''):\n            digest.update(block)\n    result = {'schema_version': SCHEMA_VERSION, 'algorithm_version': ALGORITHM_VERSION,\n              'source': {'name': path.name, 'sha256': digest.hexdigest(), **metadata},\n              'config': {**asdict(config), 'smoothing_weights': list(KERNELS[config.smoothing])},\n              'window': {'input_start_seconds': start_seconds, 'input_end_seconds': output_end,\n                         'duration_seconds': round(output_end - start_seconds, 9),\n                         'unscored_prefix_seconds': max(0, target_samples[0]['input_time_seconds'] - start_seconds)},\n              'score': float(np.mean(valid)), 'peak_score': float(max(valid)),\n              'calibrated_sample_mean': float(np.mean([s['calibrated_score'] for s in target_samples])),\n              'sample_count': len(target_samples), 'scores': [r['score'] for r in rows],\n              'seconds': rows, 'elapsed_seconds': time.perf_counter() - started}\n    if include_samples:\n        result['samples'] = [{**s, 'time_seconds': round(s['input_time_seconds'] - start_seconds, 6)}\n                             for s in target_samples]\n    return result\n\n\ndef export_plot(result, png_path=None, html_path=None):\n    \"\"\"Optional Matplotlib exports; HTML is self-contained and embeds exact JSON.\"\"\"\n    try:\n        import matplotlib\n        matplotlib.use('Agg')\n        import matplotlib.pyplot as plt\n    except ImportError as error:\n        raise RuntimeError('Plot export requires: pip install -r requirements-plot.txt') from error\n    rows = result['seconds']\n    x = [(r['start_seconds'] + r['end_seconds']) / 2 for r in rows]\n    y = [r['score'] if r['score'] is not None else np.nan for r in rows]\n    raw = [r['calibrated_score'] if r['calibrated_score'] is not None else np.nan for r in rows]\n    fig, ax = plt.subplots(figsize=(14, 5))\n    fig.subplots_adjust(left=.07, right=.98, bottom=.2, top=.85)\n    ax.plot(x, raw, color='#a5acb5', lw=1, label='Calibrated, unsmoothed')\n    ax.plot(x, y, color='#d17715', lw=1.8, label=f\"Final: {result['config']['smoothing']}\")\n    ax.fill_between(x, y, color='#d17715', alpha=.12)\n    ax.set(xlim=(0, result['window']['duration_seconds']), ylim=(-.02, 1.02),\n           xlabel='Time relative to requested window (seconds)', ylabel='Motion score (0–1)')\n    prefix = result['window']['unscored_prefix_seconds']\n    if prefix:\n        ax.axvspan(0, prefix, color='#64748b', alpha=.15)\n    ax.set_title(f\"{result['source']['name']} — motion graph\", loc='left', fontsize=17)\n    ax.grid(alpha=.15)\n    ax.legend(loc='upper right')\n    fig.text(.07, .04, f\"{ALGORITHM_VERSION} | Mean {result['score']:.4f} | Peak {result['peak_score']:.4f} | Fixed calibration; no per-video normalization\", fontsize=10)\n    if png_path:\n        png_path = Path(png_path)\n        png_path.parent.mkdir(parents=True, exist_ok=True)\n        fig.savefig(png_path, dpi=150, facecolor='white')\n    if html_path:\n        svg = io.StringIO()\n        fig.savefig(svg, format='svg', facecolor='white')\n        content = svg.getvalue()\n        content = content[content.index('<svg'):]\n        payload = json.dumps(result, allow_nan=False).replace('</', '<\\\\/')\n        page = '''<!doctype html><html lang=\"en\"><meta charset=\"utf-8\"><meta name=\"viewport\" content=\"width=device-width,initial-scale=1\"><title>Motion graph</title><style>body{font:16px system-ui;margin:24px;color:#172435}main{max-width:1450px;margin:auto}svg{width:100%;height:auto}button{padding:12px}p{line-height:1.6}</style><main><h1>Motion graph</h1>''' + content + '''<p>Gray: calibrated scores before smoothing. Orange: final scores. Shaded initial coverage lacks comparison context. Source timestamps and per-second coverage are included in the downloadable results. This page works offline.</p><button id=\"download\">Download results JSON</button></main><script id=\"data\" type=\"application/json\">''' + payload + '''</script><script>const r=JSON.parse(document.getElementById('data').textContent);document.getElementById('download').addEventListener('click',()=>{const u=URL.createObjectURL(new Blob([JSON.stringify(r,null,2)],{type:'application/json'})),a=document.createElement('a');a.href=u;a.download='motion-results.json';a.click();setTimeout(()=>URL.revokeObjectURL(u),1000)});</script></html>'''\n        html_path = Path(html_path)\n        html_path.parent.mkdir(parents=True, exist_ok=True)\n        html_path.write_text(page, encoding='utf-8')\n    plt.close(fig)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument('video', type=Path)\n    parser.add_argument('--output', type=Path, help='JSON path; otherwise print JSON to stdout')\n    parser.add_argument('--plot', type=Path, help='Optional PNG output')\n    parser.add_argument('--html', type=Path, help='Optional self-contained HTML output')\n    parser.add_argument('--smoothing', choices=KERNELS, default='stronger')\n    parser.add_argument('--start-seconds', type=float, default=0)\n    parser.add_argument('--end-seconds', type=float)\n    parser.add_argument('--include-samples', action='store_true')\n    args = parser.parse_args()\n    try:\n        # Never overwrite a recording with a result or collide export paths.\n        outputs = [p for p in (args.output, args.plot, args.html) if p is not None]\n        resolved = [p.resolve() for p in outputs]\n        if args.video.resolve() in resolved or len(set(resolved)) != len(resolved):\n            raise ValueError('Input and output paths must be distinct')\n        result = analyze_video(args.video, Config(smoothing=args.smoothing),\n                               start_seconds=args.start_seconds, end_seconds=args.end_seconds,\n                               include_samples=args.include_samples)\n        if args.plot or args.html:\n            export_plot(result, args.plot, args.html)\n        text = json.dumps(result, indent=2, allow_nan=False)\n        if args.output:\n            args.output.parent.mkdir(parents=True, exist_ok=True)\n            args.output.write_text(text + '\\n', encoding='utf-8')\n        else:\n            print(text)\n        return 0\n    except (ValueError, RuntimeError, OSError) as error:\n        print(f'motion_graph: {error}', file=sys.stderr)\n        return 1\n\n\nif __name__ == '__main__':\n    raise SystemExit(main())\n"
}
for filename, content in motion_sources.items():
    (motion_code_dir / filename).write_text(content, encoding='utf-8')
for name in ('smoothing', 'motion_graph'):
    spec = importlib.util.spec_from_file_location(name, motion_code_dir / f'{name}.py')
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
motion_graph = sys.modules['motion_graph']


In [ ]:
# Analyze the original recording once, preserving continuity across 10s boundaries.
motion_result = motion_graph.analyze_video(long_input)

motion_seconds = motion_result['seconds']
motion_times = [(r['start_seconds'] + r['end_seconds']) / 2 for r in motion_seconds]
motion_scores = [r['score'] if r['score'] is not None else np.nan for r in motion_seconds]
fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(motion_times, motion_scores, color='#d17715', linewidth=1.7)
ax.fill_between(motion_times, motion_scores, color='#d17715', alpha=.12)
ax.axvspan(0, motion_result['window']['unscored_prefix_seconds'], color='gray', alpha=.15)
ax.set(xlim=(0, motion_result['window']['duration_seconds']), ylim=(-.02, 1.02),
       xlabel='Video time (seconds)', ylabel='Motion score (0–1)',
       title=f'Motion graph: {long_name}')
ax.grid(alpha=.15)
plt.tight_layout()
fig.savefig(long_dir / 'motion_graph.png', dpi=150)
plt.show()

print(f"Mean: {motion_result['score']:.4f} | Peak: {motion_result['peak_score']:.4f}")
print('Initial gray region: no preceding comparison context. Default: smooth calibration + stronger smoothing.')
long_output_with_motion = {**long_output, 'motion_graph': motion_result,
                           'motion_source_commit': MOTION_SOURCE_COMMIT}
combined_json_path = long_dir / 'long_video_with_motion.json'
combined_json_path.write_text(json.dumps(long_output_with_motion, indent=2))
display(JSON(long_output_with_motion, expanded=False))
files.download(str(combined_json_path))
